In [ ]:
!pip install -q scikit-learn pandas numpy joblib

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score

print('✅ All libraries imported!')
# Download the dataset directly — no manual upload needed
import urllib.request

urllib.request.urlretrieve(
    'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv',
    'Telco-Customer-Churn.csv'
)

df = pd.read_csv('Telco-Customer-Churn.csv')
print(f'Shape: {df.shape}')

# Drop ID column
df.drop(columns=['customerID'], inplace=True)

# TotalCharges is stored as string — convert to number
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'].fillna(0, inplace=True)

# Convert target: Yes → 1, No → 0
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

print('✅ Data cleaned!')
print(df['Churn'].value_counts())

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y       # keeps churn ratio balanced in both splits
)

print(f'Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows')
# Identify column types
numerical_cols   = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object']).columns.tolist()

print('Numerical  :', numerical_cols)
print('Categorical:', categorical_cols)

# Numerical: fill nulls → scale
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Categorical: fill nulls → one-hot encode
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine both
preprocessor = ColumnTransformer([
    ('num', numerical_pipeline,   numerical_cols),
    ('cat', categorical_pipeline, categorical_cols)
])

print('✅ Preprocessor ready!')
lr_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)
lr_preds = lr_pipeline.predict(X_test)

print('=== Logistic Regression ===')
print(f'Accuracy : {accuracy_score(y_test, lr_preds):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, lr_pipeline.predict_proba(X_test)[:,1]):.4f}')
print(classification_report(y_test, lr_preds, target_names=['No Churn', 'Churn']))
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

rf_pipeline.fit(X_train, y_train)
rf_preds = rf_pipeline.predict(X_test)

print('=== Random Forest ===')
print(f'Accuracy : {accuracy_score(y_test, rf_preds):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, rf_pipeline.predict_proba(X_test)[:,1]):.4f}')
print(classification_report(y_test, rf_preds, target_names=['No Churn', 'Churn']))
# Tune Logistic Regression
# Note: use 'model__param' syntax to access params inside a Pipeline
lr_param_grid = {
    'model__C'       : [0.01, 0.1, 1, 10],
    'model__penalty' : ['l1', 'l2'],
    'model__solver'  : ['liblinear']
}

lr_grid = GridSearchCV(lr_pipeline, lr_param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
lr_grid.fit(X_train, y_train)

print('=== LR Best Params ===')
print(lr_grid.best_params_)
print(f'Best CV ROC-AUC: {lr_grid.best_score_:.4f}')

# Tune Random Forest
rf_param_grid = {
    'model__n_estimators'    : [100, 200],
    'model__max_depth'       : [None, 5, 10],
    'model__min_samples_split': [2, 5]
}

rf_grid = GridSearchCV(rf_pipeline, rf_param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
rf_grid.fit(X_train, y_train)

print('\n=== RF Best Params ===')
print(rf_grid.best_params_)
print(f'Best CV ROC-AUC: {rf_grid.best_score_:.4f}')
def evaluate(name, model):
    preds  = model.predict(X_test)
    proba  = model.predict_proba(X_test)[:, 1]
    print(f'\n=== {name} ===')
    print(f'  Accuracy : {accuracy_score(y_test, preds):.4f}')
    print(f'  ROC-AUC  : {roc_auc_score(y_test, proba):.4f}')
    return roc_auc_score(y_test, proba)

lr_auc = evaluate('Logistic Regression (Tuned)', lr_grid.best_estimator_)
rf_auc = evaluate('Random Forest (Tuned)',        rf_grid.best_estimator_)

best_pipeline = rf_grid.best_estimator_ if rf_auc >= lr_auc else lr_grid.best_estimator_
winner = 'Random Forest' if rf_auc >= lr_auc else 'Logistic Regression'
print(f'\n🏆 Best Model: {winner}')
# Save the full pipeline (preprocessing + model) to disk
joblib.dump(best_pipeline, 'churn_pipeline.pkl')
print('✅ Saved: churn_pipeline.pkl')

# Reload and verify it works
loaded = joblib.load('churn_pipeline.pkl')
sample = X_test.iloc[:5]
preds  = loaded.predict(sample)
probs  = loaded.predict_proba(sample)[:, 1]

print('\nSample Predictions:')
for i, (p, prob) in enumerate(zip(preds, probs)):
    label = 'Churn ⚠️' if p == 1 else 'No Churn ✅'
    print(f'  Customer {i+1}: {label}  ({prob:.2%} probability)')